In [1]:
import polars as pl
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

INPUT_PATH  = "stgcn_dataset/node_features_X.parquet"
OUTPUT_PATH = "xgboost_dataset/xgb_tabular.parquet"

In [2]:
LAG_BASE_COLS = [
    "demand",
    "revenue_total",
]

LAG_WEATHER_COLS = [
    "temperature",
    "wind_speed",
    "precipitation",
]

ROLLING_SUMMARY_COLS = [
    "rolling_tip_pct",
    "rolling_avg_fare",
    "rolling_peak_ratio",
]

CALENDAR_COLS = [
    "hour",
    "weekday",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "is_holiday",
]

GEO_COLS = [
    "zone_area_sqkm",
    "dist_to_center_km",
]

ID_COLS = [
    "time_bin",
    "LocationID",
]

LAGS_BASE = [0, 1, 2, 3, 23, 167]
LAGS_WEATHER = [0, 23, 167]

In [3]:
raw_df = pl.read_parquet(INPUT_PATH)

In [4]:
df = raw_df.filter(pl.col("time_bin").dt.year() >= 2023)
df = df.sort(["LocationID", "time_bin"])

lag_exprs_base = []
for col in LAG_BASE_COLS:
    for lag in LAGS_BASE:
        lag_exprs_base.append(
            pl.col(col)
            .shift(lag)
            .over("LocationID")
            .alias(f"{col}_lag_{lag}")
        )

lag_exprs_weather = []
for col in LAG_WEATHER_COLS:
    for lag in LAGS_WEATHER:
        lag_exprs_weather.append(
            pl.col(col)
            .shift(lag)
            .over("LocationID")
            .alias(f"{col}_lag_{lag}")
        )

target_exprs = [
    pl.col("demand").shift(-1).over("LocationID").alias("target_demand"),
    pl.col("revenue_total").shift(-1).over("LocationID").alias("target_revenue"),
]

df_final = df.select(
    *ID_COLS,
    *GEO_COLS,
    *CALENDAR_COLS,
    *ROLLING_SUMMARY_COLS,
    *lag_exprs_base,
    *lag_exprs_weather,
    *target_exprs,
)

df_final = df_final.with_columns([
    pl.when(pl.col(c).is_finite())
      .then(pl.col(c))
      .otherwise(None)
      .alias(c)
    for c in ROLLING_SUMMARY_COLS
])

df_final = df_final.drop_nulls()

In [5]:
df_final.write_parquet(OUTPUT_PATH)
print(f"\nSaved XGBoost dataset to {OUTPUT_PATH}")


Saved XGBoost dataset to xgboost_dataset/xgb_tabular.parquet


In [6]:
df_final

time_bin,LocationID,zone_area_sqkm,dist_to_center_km,hour,weekday,hour_sin,hour_cos,weekday_sin,weekday_cos,is_holiday,rolling_tip_pct,rolling_avg_fare,rolling_peak_ratio,demand_lag_0,demand_lag_1,demand_lag_2,demand_lag_3,demand_lag_23,demand_lag_167,revenue_total_lag_0,revenue_total_lag_1,revenue_total_lag_2,revenue_total_lag_3,revenue_total_lag_23,revenue_total_lag_167,temperature_lag_0,temperature_lag_23,temperature_lag_167,wind_speed_lag_0,wind_speed_lag_23,wind_speed_lag_167,precipitation_lag_0,precipitation_lag_23,precipitation_lag_167,target_demand,target_revenue
datetime[μs],i64,f64,f64,i8,i8,f64,f64,f64,f64,i32,f32,f64,f64,u32,u32,u32,u32,u32,u32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,u32,f32
2023-01-07 23:00:00,1,7.343009,17.611394,23,6,-0.258819,0.965926,-0.781831,0.62349,0,0.131269,91.908439,0.403922,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,5.6,6.1,10.0,3.1,4.6,0.25,0.0,0.0,1.5,0,0.0
2023-01-08 00:00:00,1,7.343009,17.611394,0,7,0.0,1.0,-2.4493e-16,1.0,0,0.131118,92.27036,0.398374,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.6,11.7,4.1,4.6,1.533333,0.0,0.0,2.0,0,0.0
2023-01-08 01:00:00,1,7.343009,17.611394,1,7,0.258819,0.965926,-2.4493e-16,1.0,0,0.131118,92.27036,0.398374,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,5.6,12.1,4.1,4.1,1.3,0.0,0.0,2.8,0,0.0
2023-01-08 02:00:00,1,7.343009,17.611394,2,7,0.5,0.866025,-2.4493e-16,1.0,0,0.131118,92.27036,0.398374,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,4.4,4.4,12.6,2.6,3.6,2.6,0.0,0.0,2.3,0,0.0
2023-01-08 03:00:00,1,7.343009,17.611394,3,7,0.707107,0.707107,-2.4493e-16,1.0,0,0.131118,92.27036,0.398374,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,4.4,3.9,12.2,2.1,3.1,1.2,0.0,6.9,7.4,0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-08-26 18:00:00,263,0.616541,3.626352,18,2,-1.0,-1.8370e-16,0.974928,-0.222521,0,0.15636,14.638287,0.974059,106,93,81,93,71,104,1635.98999,1711.790039,1119.97998,1478.450073,1228.47998,1397.280029,25.6,28.300001,23.300001,3.1,2.1,3.6,0.0,0.0,0.0,76,1064.390015
2025-08-26 19:00:00,263,0.616541,3.626352,19,2,-0.965926,0.258819,0.974928,-0.222521,0,0.15636,14.638287,0.974059,76,106,93,81,81,96,1064.390015,1635.98999,1711.790039,1119.97998,1196.150024,1394.51001,24.4,27.200001,23.300001,3.1,2.1,3.6,0.0,0.0,0.0,80,1208.660034
2025-08-26 20:00:00,263,0.616541,3.626352,20,2,-0.866025,0.5,0.974928,-0.222521,0,0.15636,14.638287,0.974059,80,76,106,93,65,96,1208.660034,1064.390015,1635.98999,1711.790039,960.799988,1434.570068,25.0,27.200001,22.200001,3.1,2.1,3.6,0.0,0.0,0.0,72,1230.530029


In [7]:
df_final = pl.read_parquet(OUTPUT_PATH)

In [7]:
TARGET_COLS = [
    "target_demand",
    "target_revenue",
]

DROP_COLS = [
    "time_bin",
]

ID_COLS = [
    "LocationID",
]

FEATURE_COLS = [
    c for c in df_final.columns
    if c not in TARGET_COLS + DROP_COLS
]

In [8]:
df_final_year = df_final.with_columns(pl.col("time_bin").dt.year().alias("year"))

train_df = df_final_year.filter(pl.col("year") == 2023)
val_df   = df_final_year.filter(pl.col("year") == 2024)
test_df  = df_final_year.filter(pl.col("year") == 2025)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

Train: (2244472, 38)
Val  : (2290368, 38)
Test : (1472652, 38)


In [9]:
X_train = train_df.select(FEATURE_COLS).to_numpy()
y_train = train_df.select(TARGET_COLS).to_numpy()

X_val = val_df.select(FEATURE_COLS).to_numpy()
y_val = val_df.select(TARGET_COLS).to_numpy()

X_test = test_df.select(FEATURE_COLS).to_numpy()
y_test = test_df.select(TARGET_COLS).to_numpy()

In [10]:
demand_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
)

demand_model.fit(
    X_train,
    y_train[:, 0],
    eval_set=[(X_val, y_val[:, 0])],
    verbose=True,
)

[0]	validation_0-rmse:51.19985
[1]	validation_0-rmse:48.79322
[2]	validation_0-rmse:46.51312
[3]	validation_0-rmse:44.36355
[4]	validation_0-rmse:42.30866
[5]	validation_0-rmse:40.36214
[6]	validation_0-rmse:38.51962
[7]	validation_0-rmse:36.77775
[8]	validation_0-rmse:35.12624
[9]	validation_0-rmse:33.55962
[10]	validation_0-rmse:32.07605
[11]	validation_0-rmse:30.68000
[12]	validation_0-rmse:29.35883
[13]	validation_0-rmse:28.10719
[14]	validation_0-rmse:26.92744
[15]	validation_0-rmse:25.81185
[16]	validation_0-rmse:24.76078
[17]	validation_0-rmse:23.76594
[18]	validation_0-rmse:22.82977
[19]	validation_0-rmse:21.94751
[20]	validation_0-rmse:21.11843
[21]	validation_0-rmse:20.33650
[22]	validation_0-rmse:19.60576
[23]	validation_0-rmse:18.91654
[24]	validation_0-rmse:18.26900
[25]	validation_0-rmse:17.66217
[26]	validation_0-rmse:17.09735
[27]	validation_0-rmse:16.56894
[28]	validation_0-rmse:16.06596
[29]	validation_0-rmse:15.60969
[30]	validation_0-rmse:15.17169
[31]	validation_0-

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [11]:
revenue_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
)

revenue_model.fit(
    X_train,
    y_train[:, 1],
    eval_set=[(X_val, y_val[:, 1])],
    verbose=True,
)

[0]	validation_0-rmse:1571.68109
[1]	validation_0-rmse:1501.87949
[2]	validation_0-rmse:1435.43480
[3]	validation_0-rmse:1372.59963
[4]	validation_0-rmse:1313.23562
[5]	validation_0-rmse:1256.82772
[6]	validation_0-rmse:1203.87917
[7]	validation_0-rmse:1154.03049
[8]	validation_0-rmse:1106.62503
[9]	validation_0-rmse:1062.21728
[10]	validation_0-rmse:1020.19732
[11]	validation_0-rmse:982.07613
[12]	validation_0-rmse:945.20983
[13]	validation_0-rmse:910.42046
[14]	validation_0-rmse:877.40283
[15]	validation_0-rmse:846.47094
[16]	validation_0-rmse:817.68326
[17]	validation_0-rmse:790.52874
[18]	validation_0-rmse:765.38951
[19]	validation_0-rmse:741.76193
[20]	validation_0-rmse:719.55105
[21]	validation_0-rmse:699.11455
[22]	validation_0-rmse:679.75520
[23]	validation_0-rmse:661.73447
[24]	validation_0-rmse:645.05345
[25]	validation_0-rmse:629.86886
[26]	validation_0-rmse:615.49704
[27]	validation_0-rmse:602.15527
[28]	validation_0-rmse:589.67480
[29]	validation_0-rmse:578.29139
[30]	vali

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [18]:
demand_model.save_model("xgboost_dataset/demand_model.json")
revenue_model.save_model("xgboost_dataset/revenue_model.json")

In [13]:
# Predictions
demand_pred = demand_model.predict(X_test)
revenue_pred = revenue_model.predict(X_test)

# Metrics
demand_mae = mean_absolute_error(y_test[:, 0], demand_pred)
demand_rmse = root_mean_squared_error(y_test[:, 0], demand_pred)

revenue_mae = mean_absolute_error(y_test[:, 1], revenue_pred)
revenue_rmse = root_mean_squared_error(y_test[:, 1], revenue_pred)

print("=" * 50)
print("XGBOOST TEST PERFORMANCE (Year 2025)")
print("=" * 50)

print("[Target 1] DEMAND")
print(f"  MAE : {demand_mae:.2f} trips")
print(f"  RMSE: {demand_rmse:.2f} trips")

print("-" * 50)

print("[Target 2] REVENUE")
print(f"  MAE : ${revenue_mae:.2f}")
print(f"  RMSE: ${revenue_rmse:.2f}")
print("=" * 50)

XGBOOST TEST PERFORMANCE (Year 2025)
[Target 1] DEMAND
  MAE : 3.29 trips
  RMSE: 9.71 trips
--------------------------------------------------
[Target 2] REVENUE
  MAE : $93.73
  RMSE: $843.13


In [16]:
# Filter to test year (2025)
df_test = df_final.filter(pl.col("time_bin").dt.year() == 2025)

# Convert to numpy arrays for demand
y_true_demand = df_test["target_demand"].to_numpy()
y_pred_demand_lag24 = df_test["demand_lag_167"].to_numpy()  # naive forecast

# Convert to numpy arrays for revenue
y_true_revenue = df_test["target_revenue"].to_numpy()
y_pred_revenue_lag24 = df_test["revenue_total_lag_167"].to_numpy()  # naive forecast

# Compute MAE and RMSE for demand
mae_demand_lag24 = mean_absolute_error(y_true_demand, y_pred_demand_lag24)
rmse_demand_lag24 = root_mean_squared_error(y_true_demand, y_pred_demand_lag24)

# Compute MAE and RMSE for revenue
mae_revenue_lag24 = mean_absolute_error(y_true_revenue, y_pred_revenue_lag24)
rmse_revenue_lag24 = root_mean_squared_error(y_true_revenue, y_pred_revenue_lag24)

print("Naive Forecast using 168h Ago Values")
print("="*50)
print("[Target 1] DEMAND")
print(f"  MAE  : {mae_demand_lag24:.2f} trips")
print(f"  RMSE : {rmse_demand_lag24:.2f} trips")
print("-"*50)
print("[Target 2] REVENUE")
print(f"  MAE  : ${mae_revenue_lag24:.2f}")
print(f"  RMSE : ${rmse_revenue_lag24:.2f}")
print("="*50)

Naive Forecast using 168h Ago Values
[Target 1] DEMAND
  MAE  : 4.75 trips
  RMSE : 15.25 trips
--------------------------------------------------
[Target 2] REVENUE
  MAE  : $129.36
  RMSE : $1180.01
